In [1]:
import numpy as np
from netCDF4 import Dataset
import matplotlib.pyplot as plt
#import cartopy.crs as crs
from cartopy.io.shapereader import Reader
from cartopy.feature import ShapelyFeature
from cartopy.feature import NaturalEarthFeature
import cartopy.crs as ccrs
from datetime import datetime, timedelta
from metpy.units import units
from metpy.calc import (dewpoint_from_specific_humidity, relative_humidity_from_specific_humidity, wind_speed, temperature_from_potential_temperature,
                    specific_humidity_from_mixing_ratio)
import pyart
import geopy.distance as distance
import haversine
from wrf import getvar
from metpy.interpolate import log_interpolate_1d


## You are using the Python ARM Radar Toolkit (Py-ART), an open source
## library for working with weather radar data. Py-ART is partly
## supported by the U.S. Department of Energy as part of the Atmospheric
## Radiation Measurement (ARM) Climate Research Facility, an Office of
## Science user facility.
##
## If you use this software to prepare a publication, please cite:
##
##     JJ Helmus and SM Collis, JORS 2016, doi: 10.5334/jors.119



In [2]:
#ncfile1 = Dataset('/glade/work/mawilson/DART_mpd/observations/utilities/threed_sphere/obs_epoch_100mIOP6.nc')
ncfile1 = Dataset('/glade/work/mawilson/DART_mpd/observations/utilities/threed_sphere/obs_epoch_100mIOP4.nc')
#ncfile1 = Dataset('/glade/work/mawilson/DART_mpd/observations/utilities/threed_sphere/obs_epoch_100mnew.nc')

location = ncfile1.variables['location'][:]
qc = ncfile1.variables['qc'][:]
obstype = ncfile1.variables['obs_type'][:]
obstypemd = ncfile1.variables['ObsTypesMetaData'][:]
obs_val = ncfile1.variables['observations'][:]
which_vert = ncfile1.variables['which_vert'][:]
times = ncfile1.variables['time'][:]
print(obstype)
qc_new = []
for i in range(len(qc)):
    qc_d = qc[i][0]
    qc_new.append(qc_d)
qc_new = np.asarray(qc_new)

[142 105 106 ...  26  27  28]


In [3]:
print(times)

[153965.99444444 153965.99444444 153965.99444444 ... 153966.41662037
 153966.41662037 153966.41662037]


In [4]:
otype_s = []
obs_s = []
lon_s = []
lat_s = []
elev_s = []
error_s = []
time_s = []

minute_range = np.arange(180,185,5)
#minute_range = np.arange(595,605,5)

for mins in minute_range:
    #dt_start = datetime(2022,9,15,0,0)
    dt_start = datetime(2022,7,19,0,0)
    #dt_start = datetime(2021,6,4,0,0)
    
    dt = dt_start + timedelta(minutes=int(mins))
    print(dt)
    dt_2 = dt - timedelta(minutes=15)
    
    #wrfout = Dataset('/glade/derecho/scratch/mawilson/20210604_newcase/nature_100IOP6/final_nature/wrfout_d02_2022-'+str(dt.isoformat()[5:7])+'-'+str(dt.isoformat()[8:10])+'_'+str(dt.isoformat()[11:13])+':'+str(dt.isoformat()[14:16])+':00')
    #wrfout = Dataset('/glade/derecho/scratch/mawilson/20210604_newcase/nature_100IOP4/final_nature/wrfout_d02_2022-'+str(dt.isoformat()[5:7])+'-'+str(dt.isoformat()[8:10])+'_'+str(dt.isoformat()[11:13])+':'+str(dt.isoformat()[14:16])+':00')
    #wrfout = Dataset('/glade/campaign/ral/aap/mawilson/nature_runs/JUNE/final_nature/wrfout_d02_2021-'+str(dt.isoformat()[5:7])+'-'+str(dt.isoformat()[8:10])+'_'+str(dt.isoformat()[11:13])+':'+str(dt.isoformat()[14:16])+':00')
    wrfout = Dataset('/glade/campaign/ral/aap/mawilson/nature_runs/IOP4/final_nature/wrfout_d02_2022-'+str(dt.isoformat()[5:7])+'-'+str(dt.isoformat()[8:10])+'_'+str(dt.isoformat()[11:13])+':'+str(dt.isoformat()[14:16])+':00')
    
    lon = wrfout.variables['XLONG']
    lat = wrfout.variables['XLAT']
    U10 = wrfout.variables['U10']
    V10 = wrfout.variables['U10']
    T2 = np.asarray(wrfout.variables['T2'])*units('K')
    T2F = T2 .to('degF')
    Q2 = np.asarray(wrfout.variables['Q2'])
    P2 = np.asarray(wrfout.variables['PSFC'][:]/100) * units('hPa')
    Td2 = dewpoint_from_specific_humidity(P2[0,:,:], T2[0,:,:], Q2[0,:,:]*units('kg/kg'))
    RH2 = relative_humidity_from_specific_humidity(P2[0,:,:], T2[0,:,:], Q2[0,:,:]*units('kg/kg'))
    SPD10 = wind_speed(np.asarray(U10)*units('m/s'), np.asarray(V10)*units('m/s'))
    cloud=wrfout.variables['QCLOUD']
    T_z = np.asarray(getvar(wrfout, "tk"))
    p_z = np.asarray(getvar(wrfout, "pres"))
    q_zi = np.asarray(wrfout.variables['QVAPOR'][:])
    q_z = specific_humidity_from_mixing_ratio(q_zi)
    u_z, v_z = getvar(wrfout, 'uvmet')
    u_z = np.asarray(u_z)
    v_z = np.asarray(v_z)
    
    
    #otype = 107
    #otype = 105
    #otype = 106
    #otype = 108
    #otype = 142
    otype_list = [18,16,17]
    for otype in otype_list:
        loc_T2 = location[obstype==otype, :]
        qc_T2 = qc_new[obstype==otype]
        obs_T2 = obs_val[obstype==otype, :]
        lons_T2 = loc_T2[:,0]
        lats_T2 = loc_T2[:,1]
        elev_T2 = loc_T2[:,2]
        time_T2 = times[obstype==otype]
        lons_T2[lons_T2 > 180] = lons_T2[lons_T2 > 180] - 360
        
        #Convert WRF file time into same units as the obs_seq time
        dt_tot = (dt_2 - datetime(1601,1,1)).total_seconds() / 86400
        time_diff = np.abs(dt_tot - time_T2)
        
        #Get obs within +- 2.5 minutes of each WRF file
        time_T3 = time_T2[time_diff<(900/86400)]
        loc_T3 = loc_T2[time_diff<(900/86400), :]
        qc_T3 = qc_T2[time_diff<(900/86400)]
        lons_T3 = lons_T2[time_diff<(900/86400)]
        lats_T3 = lats_T2[time_diff<(900/86400)]
        elev_T3 = elev_T2[time_diff<(900/86400)]
        obs_T3 = obs_T2[time_diff<(900/86400), :]
        
        if len(obs_T3)==0:
            print('NO OBS IN WINDOW')
        for k in range(len(lons_T3)):
            latp=lats_T3[k]
            lonp=lons_T3[k]
            #Get location for each ob in model land
            lon1d = np.ndarray.flatten(lon[0,:,:])
            lat1d = np.ndarray.flatten(lat[0,:,:])
            station = []
            points = []
            for i in range(len(lon1d)):
                points.append((lat1d[i],lon1d[i]))
                station.append((latp,lonp))
            dist = haversine.haversine_vector(station,points)
            dist2=dist.reshape(lon.shape[1],lon.shape[2])
            print(lon[0,:,:][np.where(dist2==np.min(dist2))])
            print(lat[0,:,:][np.where(dist2==np.min(dist2))])
            print(np.where(dist2==np.min(dist2)))
            st_xind = np.where(dist2==np.min(dist2))[0][0]
            st_yind = np.where(dist2==np.min(dist2))[1][0]
            print(elev_T3[k], 'elev')
            
            if otype == 18:
                p_point = p_z[:,st_xind,st_yind]
                t_point = T_z[:,st_xind,st_yind]
                T2_a = log_interpolate_1d(elev_T3[k], p_point, t_point)
                
            elif otype == 16:
                p_point = p_z[:,st_xind,st_yind]
                t_point = u_z[:,st_xind,st_yind]
                T2_a = log_interpolate_1d(elev_T3[k], p_point, t_point)
                
            elif otype == 17:
                p_point = p_z[:,st_xind,st_yind]
                t_point = v_z[:,st_xind,st_yind]
                T2_a = log_interpolate_1d(elev_T3[k], p_point, t_point)

            if np.isnan(T2_a):
                #If the observation is a nan or outside of the the interpolation bounds, skip it
                print('skipping nan ob')
                continue
            
            #If you want to change the error assumption, just change the scale in this line
            error = np.random.normal(loc=0.0, scale=np.sqrt(obs_T3[k,1]))
            #6/17/2024 MW adding code to limit added error to 1.5 times the error assumtion in DART
            if np.abs(error/4) > (np.sqrt(obs_T3[k,1])*1.0):
                error = (error / np.abs(error)) * (np.sqrt(obs_T3[k,1])*1.0)
            T2_b = T2_a + (error/4)
            print(T2_a, (error/4))
        
            #Append obs to arrays for writing to obs_seq file later
            otype_s.append(otype)
            obs_s.append(T2_b)
            lon_s.append(lonp)
            lat_s.append(latp)
            elev_s.append(elev_T3[k])
            error_s.append(obs_T3[k,1]/2)
            time_s.append(time_T3[k])

2022-07-19 03:00:00
[-83.89059]
[38.838234]
(array([174]), array([1200]))
22710.70471054592 elev
[228.81261144] -0.1253438444812471
[-84.87757]
[38.78215]
(array([111]), array([431]))
57006.14134259468 elev
[271.7763483] 0.16035593026164036
[-84.8018]
[38.802906]
(array([132]), array([490]))
54233.47028483736 elev
[269.60658686] -0.015891945955577427
[-84.68309]
[39.433075]
(array([763]), array([579]))
37093.9873054299 elev
[251.18012364] 0.05670532389831505
[-84.99957]
[38.707474]
(array([36]), array([336]))
61311.38209971337 elev
[275.18189755] 0.3158434966150861
[-84.94946]
[38.753353]
(array([82]), array([375]))
59672.98757682822 elev
[273.99472302] -0.3387990963850912
[-84.70032]
[39.373184]
(array([703]), array([566]))
37645.93513721537 elev
[251.90555756] 0.06897797013479177
[-84.429825]
[38.810043]
(array([141]), array([780]))
67270.4617176398 elev
[278.36014135] 0.3195156379144738
[-84.429825]
[38.810043]
(array([141]), array([780]))
67267.70077766982 elev
[278.35873746] 0.580

/glade/derecho/scratch/mawilson/tmp/ipykernel_88579/3876484403.py:101: UserWarning: Interpolation point out of data bounds encountered
  T2_a = log_interpolate_1d(elev_T3[k], p_point, t_point)


[-84.958435]
[38.750366]
(array([79]), array([368]))
59696.47647384142 elev
[274.02671936] 0.17490242736798706
[-84.91341]
[38.797245]
(array([126]), array([403]))
57422.27666427687 elev
[272.08147768] 0.017202897102082032
[-84.66021]
[39.0402]
(array([370]), array([599]))
97304.56747627366 elev
[297.82800397] 0.2513548693785579
[-84.6603]
[39.03021]
(array([360]), array([599]))
95080.51009476722 elev
[296.79140365] 0.24452961876268317
[-85.00213]
[38.705486]
(array([34]), array([334]))
61191.44526390271 elev
[275.11414554] 0.2296178292250666
[-84.66037]
[39.020218]
(array([350]), array([599]))
92931.41459350563 elev
[295.70221242] -0.1737229106898622
[-84.66044]
[39.010223]
(array([340]), array([599]))
91196.45722974424 elev
[294.44590979] 0.38762456893668784
[-84.66051]
[39.00022]
(array([330]), array([599]))
90065.38710866169 elev
[293.57569051] 0.04636298831766652
[-84.66061]
[38.990234]
(array([320]), array([599]))
89032.26353461381 elev
[292.6809041] -0.12235804632062194
[-84.659

/glade/derecho/scratch/mawilson/tmp/ipykernel_88579/3876484403.py:106: UserWarning: Interpolation point out of data bounds encountered
  T2_a = log_interpolate_1d(elev_T3[k], p_point, t_point)


[-84.958435]
[38.750366]
(array([79]), array([368]))
59696.47647384142 elev
[-0.83570144] -0.840185098484509
[-84.91341]
[38.797245]
(array([126]), array([403]))
57422.27666427687 elev
[-2.03699802] 0.7097234409172665
[-84.66021]
[39.0402]
(array([370]), array([599]))
97304.56747627366 elev
[4.1101727] -0.15949156781462298
[-84.6603]
[39.03021]
(array([360]), array([599]))
95080.51009476722 elev
[3.9283976] -0.051119495950662674
[-85.00213]
[38.705486]
(array([34]), array([334]))
61191.44526390271 elev
[-0.20558383] 0.26600952356471624
[-84.66037]
[39.020218]
(array([350]), array([599]))
92931.41459350563 elev
[2.98496803] -0.17872295796822363
[-84.66044]
[39.010223]
(array([340]), array([599]))
91196.45722974424 elev
[2.29569783] 1.6028390240875767
[-84.66051]
[39.00022]
(array([330]), array([599]))
90065.38710866169 elev
[2.10558013] -0.3902482439766631
[-84.66061]
[38.990234]
(array([320]), array([599]))
89032.26353461381 elev
[2.1066699] 1.4663851353757393
[-84.659386]
[38.98023]
(

/glade/derecho/scratch/mawilson/tmp/ipykernel_88579/3876484403.py:111: UserWarning: Interpolation point out of data bounds encountered
  T2_a = log_interpolate_1d(elev_T3[k], p_point, t_point)


[-84.958435]
[38.750366]
(array([79]), array([368]))
59696.47647384142 elev
[-3.17653957] 0.45890957052991777
[-84.91341]
[38.797245]
(array([126]), array([403]))
57422.27666427687 elev
[-3.50282146] -0.3825317668525621
[-84.66021]
[39.0402]
(array([370]), array([599]))
97304.56747627366 elev
[-1.09756747] -0.13294214991860995
[-84.6603]
[39.03021]
(array([360]), array([599]))
95080.51009476722 elev
[-0.96915453] 0.42215172459093486
[-85.00213]
[38.705486]
(array([34]), array([334]))
61191.44526390271 elev
[-3.07914182] 1.2892858569336607
[-84.66037]
[39.020218]
(array([350]), array([599]))
92931.41459350563 elev
[-0.27125765] -0.4462168232225945
[-84.66044]
[39.010223]
(array([340]), array([599]))
91196.45722974424 elev
[-0.67466732] -0.7059188180171359
[-84.66051]
[39.00022]
(array([330]), array([599]))
90065.38710866169 elev
[-1.14938346] -0.5623053748491273
[-84.66061]
[38.990234]
(array([320]), array([599]))
89032.26353461381 elev
[-1.52268727] 0.23269916767376939
[-84.659386]
[38

In [5]:
#Convert lats and lons to radians for DART, because why not
lon_DART = np.radians(np.asarray(lon_s))
lat_DART = np.radians(np.asarray(lat_s))

lon_DART = np.where(lon_DART > 0.0, lon_DART, lon_DART+(2.0*np.pi))

#Convert time into DART format. This is hacky now, improve later
#day_DART = 154024
day_DART = 153966
#day_DART = 153556

seconds_DART = (np.asarray(time_s) - day_DART) * 86400

In [6]:
#Sort everything in time order
inds_time = seconds_DART.argsort()
print(seconds_DART)
print(seconds_DART[inds_time])
seconds_DART1 = seconds_DART[inds_time]
seconds_DART1[seconds_DART1 < 0] = 0
obs_s1 = np.asarray(obs_s)[inds_time]
lon_DART1 = lon_DART[inds_time]
lat_DART1 = lat_DART[inds_time]
elev_s1 = np.asarray(elev_s)[inds_time]
otype_s1 = np.asarray(otype_s)[inds_time]
error_s1 = np.asarray(error_s)[inds_time]

[ 9033.00000122  9060.00000061  9060.00000061  9060.00000061
  9119.99999955  9119.99999955  9119.99999955  9139.9999992
  9139.9999992   9180.00000101  9239.99999994  9239.99999994
  9239.99999994  9299.99999888  9299.99999888  9360.00000034
  9360.00000034  9419.99999927  9419.99999927  9419.99999927
  9480.00000073  9600.00000112  9600.00000112  9660.00000006
  9660.00000006  9719.99999899  9719.99999899  9719.99999899
  9780.00000045  9780.00000045  9826.99999949  9838.99999978
  9839.99999939  9851.00000007  9863.00000036  9875.00000065
  9887.00000094  9893.00000109  9899.00000123  9900.00000084
  9904.99999886  9910.99999901  9932.00000077  9952.00000042
  9959.99999978  9972.00000007  9991.99999971 10011.99999936
 10031.99999901 10052.00000117 10072.00000081 10092.00000046
 10112.0000001  10139.99999911 10320.00000095 10362.99999906
 10362.99999906 10439.99999883 10620.00000067 10620.00000067
 10679.99999961 10679.99999961 10679.99999961 10740.00000106
 10740.00000106 10740.000

In [7]:
for bigfoot in [1,2]:
    print(bigfoot)

    #Write the simulated obs out to an obs_seq file
    filename = 'SIM_ACARS_IOP4_final03'
    #filename = 'SIM_ACARS_JUNE_final03'
    
    #filename = 'SIM_ACARS_IOP4_finalhalferr'
    #filename = 'SIM_ACARS_NEW_finalhalferr'
    
    fi = open(filename, "w")
    fi.write(" obs_sequence\n")
    fi.write("obs_kind_definitions\n")
    fi.write("    %d \n" %(3))
    fi.write("    %d          %s   \n" %(18, 'ACARS_TEMPERATURE'))
    fi.write("    %d          %s   \n" %(16, 'ACARS_U_WIND_COMPONENT'))
    fi.write("    %d          %s   \n" %(17, 'ACARS_V_WIND_COMPONENT'))
    
    fi.write("  num_copies:            %d  num_qc:            %d\n" % (1,1))
    fi.write(" num_obs:       %d  max_num_obs:       %d\n" % (len(obs_s1), len(obs_s1)))
    fi.write("MADIS observation\n")
    fi.write("Data QC\n")
    
    fi.write("  first:            %d  last:       %d\n" % (1, len(obs_s1)))
    
    for q in range(len(obs_s1)):
    
        fi.write(" OBS            %d\n" % (q+1) )
        fi.write("   %20.14f\n" % obs_s1[q] )
        fi.write("   %20.14f\n" % 1.0 )
    
        if q+1 == 1:
            fi.write(" %d %d %d\n" % (-1, q+2, -1) ) #First obs
        elif q+1 == len(obs_s1):
            fi.write(" %d %d %d\n" % (q, -1, -1) ) #Last obs
        else:
            fi.write(" %d %d %d\n" % (q, q+2, -1) )
    
        fi.write("obdef\n")
        fi.write("loc3d\n")
        fi.write("    %20.14f          %20.14f          %20.14f     %d\n" %
                           (lon_DART1[q], lat_DART1[q], elev_s1[q], 2))
        fi.write("kind\n")
    
        fi.write("     %d     \n" % otype_s1[q])
    
        fi.write("    %d          %d     \n" % (seconds_DART1[q], day_DART))
    
        fi.write("    %20.14f  \n" % error_s1[q])

1
2


/glade/derecho/scratch/mawilson/tmp/ipykernel_88579/2188319505.py:29: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  fi.write("   %20.14f\n" % obs_s1[q] )


In [8]:
print(obs_s)

[array([228.68726759]), array([271.93670423]), array([269.59069491]), array([251.23682896]), array([275.49774105]), array([273.65592392]), array([251.97453553]), array([278.67965698]), array([278.93943443]), array([275.80066469]), array([242.54791493]), array([263.25294591]), array([261.53566991]), array([266.74388246]), array([264.84121564]), array([269.87923219]), array([268.37774949]), array([247.93840365]), array([274.1077729]), array([272.60277953]), array([275.2008459]), array([262.73619953]), array([261.50902534]), array([264.97553194]), array([262.85561494]), array([270.22002378]), array([269.39069574]), array([267.73533195]), array([274.20162179]), array([272.09868057]), array([298.07935884]), array([297.03593327]), array([275.34376337]), array([295.52848951]), array([294.83353436]), array([293.62205349]), array([292.55854606]), array([292.41970982]), array([291.71062893]), array([256.50709371]), array([291.02317917]), array([290.48533053]), array([287.50453749]), array([284.8